# BIXI Analytics - Phase 1: Quick Exploration

## Overview
This notebook implements Phase 1 of the analysis strategy: Quick Exploration of the BIXI bike-sharing dataset.

**Phase 1 Goals:**
- Load parquet data with DuckDB
- Overview of dataset dimensions and structure
- Sample data inspection
- Initial data quality checks

In [0]:
import duckdb
import pandas as pd
from pathlib import Path

print("DuckDB version:", duckdb.__version__)
print("Libraries loaded successfully!")

## 1. Data Overview

Query the dataset to understand:
- Total number of trips
- Number of columns and their data types
- Date range of the data

In [0]:
# Get total row count
result = duckdb.sql("SELECT COUNT(*) as total_trips FROM read_parquet('data/**/*.parquet')")
print("Dataset Statistics:")
print(result.show())

# Get column information
print("\n" + "="*80)
print("Column Information:")
print("="*80)
sample_df = duckdb.sql("SELECT * FROM read_parquet('data/**/*.parquet') LIMIT 1").to_df()
print(f"{'Column Name':<35} {'Data Type'}")
print("-" * 55)
for col_name, dtype in sample_df.dtypes.items():
    print(f"{col_name:<35} {dtype}")
print(f"\nTotal columns: {len(sample_df.columns)}")

In [0]:
# Get temporal range
date_range = duckdb.sql("""
    SELECT 
        MIN(to_timestamp(start_time_ms/1000.0)) as earliest_trip,
        MAX(to_timestamp(start_time_ms/1000.0)) as latest_trip
    FROM read_parquet('data/**/*.parquet')
""")
print("Date Range:")
print(date_range.show())

## 2. Sample Data Inspection

Display the first 5 trips to understand data structure and values.

In [0]:
# Get sample data
sample = duckdb.sql("""
    SELECT * FROM read_parquet('data/**/*.parquet') 
    LIMIT 5
""").to_df()

print("First 5 trips:")
print(sample.to_string())
print(f"\nDataFrame shape: {sample.shape}")
print(f"\nData types:\n{sample.dtypes}")

## 3. Data Quality: Missing Values

Check for NULL values in each column to assess data completeness.

In [0]:
# Check for missing values
missing_values = sample.isna().sum()
print("Missing values in sample:")
print(missing_values[missing_values > 0])
if missing_values.sum() == 0:
    print("✓ No missing values in sample data")

# Full dataset missing values check via DuckDB
print("\n" + "="*80)
print("Dataset-wide NULL count by column:")
print("="*80)

# Get total row count first
total_rows = duckdb.sql("SELECT COUNT(*) as count FROM read_parquet('data/**/*.parquet')").to_df()['count'][0]
print(f"Total rows in dataset: {total_rows:,}\n")

# Get all columns and check nulls for each
all_columns = duckdb.sql("SELECT * FROM read_parquet('data/**/*.parquet') LIMIT 0").columns
print(f"{'Column Name':<40} {'Non-Nulls':>12} {'Nulls':>12} {'Null %':>10}")
print("-" * 80)

for col in all_columns:
    # Count non-null values for this column
    non_null_count = duckdb.sql(f"""
        SELECT COUNT("{col}") as count
        FROM read_parquet('data/**/*.parquet')
    """).to_df()['count'][0]
    
    null_count = total_rows - non_null_count
    null_pct = (null_count / total_rows) * 100 if total_rows > 0 else 0
    
    print(f"{col:<40} {non_null_count:>12,} {null_count:>12,} {null_pct:>9.2f}%")

print("\n✓ NULL check complete")

## Phase 1 Summary

**Phase 1: Quick Exploration ✓ Complete**

This phase has successfully:
1. ✓ Loaded DuckDB and verified connectivity to parquet data
2. ✓ Inspected dataset dimensions (row count, column count)
3. ✓ Verified data types and schema
4. ✓ Confirmed temporal range of data
5. ✓ Sampled data to understand structure
6. ✓ Checked data quality (missing values)

**Next Steps:**
- Phase 2: DuckDB-Heavy Analysis (Temporal, Geographic, Duration, Network analyses)
- Phase 3: Python Computation (Distance calculations, Outlier detection)
- Phase 4: Visualization (Heatmaps, Histograms, Scatter plots)